# C-TimeGAN Training Notebook for RunPod

This notebook is optimized for cloud training on RunPod with GPU support.
Features include:
- Auto GPU/CPU detection
- Checkpoint management for distributed training
- Periodic evaluation and metric logging
- Graceful interruption handling

## Step 1: Environment Setup and Dependencies

In [ ]:
# Navigate to project directory
import os

# Update these paths to match your Google Drive structure
COLAB_PROJECT_PATH = "/content/drive/MyDrive/NITK-Internship-Project"  # Change to your actual path
RUNPOD_PROJECT_PATH = "/workspace"

# Auto-detect environment
if 'google.colab' in sys.modules:
    PROJECT_PATH = COLAB_PROJECT_PATH
    os.chdir(PROJECT_PATH)
    print(f"Changed directory to: {PROJECT_PATH}")
else:
    PROJECT_PATH = RUNPOD_PROJECT_PATH
    print(f"RunPod environment detected. Using: {PROJECT_PATH}")

print(f"Current working directory: {os.getcwd()}")
print(f"Files in current directory: {os.listdir('.')[:10]}")

In [ ]:
# Mount Google Drive (for Colab)
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted at /content/drive")
else:
    print("Not running in Colab. Skipping Google Drive mount.")

## Step 0: Mount Google Drive (Colab Only)

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'tqdm',
    'pandas',
    'numpy',
    'scikit-learn'
]

print("Installing/verifying required packages...")
for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

In [ ]:
import os
import sys
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import grad
from tqdm import tqdm
import time
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")
    logger.info(f"Available Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Import Models from Repository

In [ ]:
# Add src/ to path for imports
sys.path.insert(0, '/workspace/src/models')
sys.path.insert(0, '/workspace/src/preprocessing')

# In Colab, module cache can keep old class definitions.
# Force reload so recent code edits are always picked up.
import importlib
import autoencoder as autoencoder_module
import generator as generator_module
import discriminator as discriminator_module
import supervisor as supervisor_module

importlib.reload(autoencoder_module)
importlib.reload(generator_module)
importlib.reload(discriminator_module)
importlib.reload(supervisor_module)

from autoencoder import SequenceAutoencoder
from generator import SequenceGenerator
from discriminator import SequenceDiscriminator
from supervisor import SequenceSupervisor

# WGAN-GP uses higher-order gradients; this avoids CuDNN RNN double-backward issues.
if torch.cuda.is_available():
    torch.backends.cudnn.enabled = False
    logger.info('Disabled CuDNN globally for this session (required for WGAN-GP + RNNs).')

logger.info("Successfully imported and reloaded model architectures from repository")

## Step 3: Define Trainer & Import Data Pipeline

In [ ]:
class RobustTrainer:
    """
    Unified, fault-tolerant trainer for C-TimeGAN.
    RunPod-optimized with cloud checkpointing.
    """
    
    def __init__(
        self,
        noise_dim=16,
        hidden_dim=24,
        input_dim=51,
        num_classes=5,
        lr=1e-4,
        continuous_indices=None,
        categorical_indices=None,
        categorical_sizes=None,
        lambda_cat=1.0,
        lambda_sup=1.0,
        checkpoint_dir=None
    ):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Initialized training on device: {self.device}")

        # Feature metadata
        self.categorical_indices = categorical_indices or []
        self.categorical_sizes = categorical_sizes or []
        if self.categorical_indices and len(self.categorical_sizes) != len(self.categorical_indices):
            raise ValueError("categorical_sizes must match categorical_indices length")
        if continuous_indices is None:
            self.continuous_indices = [i for i in range(input_dim) if i not in self.categorical_indices]
        else:
            self.continuous_indices = continuous_indices

        self.lambda_cat = lambda_cat
        self.lambda_sup = lambda_sup
        self.mse_loss = nn.MSELoss()
        self.ce_loss = nn.CrossEntropyLoss()

        # Initialize models
        self.autoencoder = SequenceAutoencoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            continuous_dim=len(self.continuous_indices),
            categorical_sizes=self.categorical_sizes
        ).to(self.device)
        self.generator = SequenceGenerator(noise_dim=noise_dim, hidden_dim=hidden_dim, num_classes=num_classes).to(self.device)
        self.discriminator = SequenceDiscriminator(hidden_dim=hidden_dim, num_classes=num_classes).to(self.device)
        self.supervisor = SequenceSupervisor(hidden_dim=hidden_dim).to(self.device)

        # Optimizers
        self.opt_G = optim.Adam(self.generator.parameters(), lr=lr, betas=(0.0, 0.9))
        self.opt_D = optim.Adam(self.discriminator.parameters(), lr=lr, betas=(0.0, 0.9))
        self.opt_AE = optim.Adam(self.autoencoder.parameters(), lr=lr)
        self.opt_S = optim.Adam(self.supervisor.parameters(), lr=lr, betas=(0.0, 0.9))

        self.noise_dim = noise_dim
        
        if checkpoint_dir is None:
            self.checkpoint_dir = "/workspace/checkpoints"  # RunPod volume mount
        else:
            self.checkpoint_dir = checkpoint_dir
        
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        self.start_epoch = 0

    def save_checkpoint(self, epoch: int, is_final: bool = False):
        """Save training state for resumption."""
        state = {
            'epoch': epoch,
            'autoencoder_state': self.autoencoder.state_dict(),
            'generator_state': self.generator.state_dict(),
            'discriminator_state': self.discriminator.state_dict(),
            'supervisor_state': self.supervisor.state_dict(),
            'opt_G_state': self.opt_G.state_dict(),
            'opt_D_state': self.opt_D.state_dict(),
            'opt_AE_state': self.opt_AE.state_dict(),
            'opt_S_state': self.opt_S.state_dict()
        }
        
        filename = 'model_final.pt' if is_final else f'checkpoint_epoch_{epoch}.pt'
        filepath = os.path.join(self.checkpoint_dir, filename)
        torch.save(state, filepath)
        torch.save(state, os.path.join(self.checkpoint_dir, 'latest_checkpoint.pt'))
        logger.info(f"Checkpoint saved at Epoch {epoch} -> {filepath}")

    def load_checkpoint(self) -> bool:
        """Load latest checkpoint to resume training."""
        latest_path = os.path.join(self.checkpoint_dir, 'latest_checkpoint.pt')
        if not os.path.exists(latest_path):
            logger.info("No existing checkpoints. Starting fresh.")
            return False
        
        logger.info(f"Loading checkpoint from {latest_path}")
        checkpoint = torch.load(latest_path, map_location=self.device)
        
        self.start_epoch = checkpoint['epoch'] + 1
        self.autoencoder.load_state_dict(checkpoint['autoencoder_state'])
        self.generator.load_state_dict(checkpoint['generator_state'])
        self.discriminator.load_state_dict(checkpoint['discriminator_state'])
        if 'supervisor_state' in checkpoint:
            self.supervisor.load_state_dict(checkpoint['supervisor_state'])
        self.opt_G.load_state_dict(checkpoint['opt_G_state'])
        self.opt_D.load_state_dict(checkpoint['opt_D_state'])
        self.opt_AE.load_state_dict(checkpoint['opt_AE_state'])
        if 'opt_S_state' in checkpoint:
            self.opt_S.load_state_dict(checkpoint['opt_S_state'])
        
        logger.info(f"Resumed from Epoch {self.start_epoch}")
        return True

    def calculate_gradient_penalty(self, real_emb, fake_emb, labels, lambda_gp=10.0):
        """WGAN-GP gradient penalty."""
        batch_size, seq_len, hidden_dim = real_emb.size()
        
        alpha = torch.rand(batch_size, 1, 1).to(self.device)
        alpha = alpha.expand_as(real_emb)
        
        interpolated = alpha * real_emb + (1 - alpha) * fake_emb
        interpolated.requires_grad_(True)
        
        prob_interpolated = self.discriminator(interpolated, labels)
        
        gradients = grad(
            outputs=prob_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones(prob_interpolated.size()).to(self.device),
            create_graph=True,
            retain_graph=True
        )[0]
        
        gradients = gradients.reshape(batch_size, -1)
        gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
        return gradient_penalty

    def compute_supervisor_loss(self, latent_seq: torch.Tensor) -> torch.Tensor:
        """Temporal coherence loss."""
        if latent_seq.size(1) < 2:
            return torch.zeros((), device=latent_seq.device)
        pred = self.supervisor(latent_seq)
        return self.mse_loss(pred[:, :-1, :], latent_seq[:, 1:, :])

    def autoencoder_step(self, real_sequences: torch.Tensor):
        """Train autoencoder step."""
        self.opt_AE.zero_grad()
        self.opt_S.zero_grad()

        if self.categorical_indices:
            recovered_cont, cat_logits, real_emb = self.autoencoder(real_sequences)
        else:
            recovered_cont, real_emb = self.autoencoder(real_sequences)
            cat_logits = []

        cont_loss = torch.zeros((), device=real_sequences.device)
        if recovered_cont is not None and self.continuous_indices:
            cont_target = real_sequences[:, :, self.continuous_indices]
            cont_loss = self.mse_loss(recovered_cont, cont_target)

        cat_loss = torch.zeros((), device=real_sequences.device)
        if self.categorical_indices:
            for head_idx, cat_idx in enumerate(self.categorical_indices):
                logits = cat_logits[head_idx]
                targets = real_sequences[:, :, cat_idx].round().long()
                cat_loss = cat_loss + self.ce_loss(
                    logits.reshape(-1, logits.size(-1)),
                    targets.reshape(-1)
                )
            cat_loss = cat_loss / len(self.categorical_indices)

        ae_loss = cont_loss + (self.lambda_cat * cat_loss)
        sup_loss = self.compute_supervisor_loss(real_emb.detach())
        total_loss = ae_loss + (self.lambda_sup * sup_loss)

        total_loss.backward()
        self.opt_AE.step()
        self.opt_S.step()

        return ae_loss, sup_loss, real_emb.detach()

    def train_step(self, real_sequences: torch.Tensor, labels: torch.Tensor, lambda_gp=10.0, n_critic=3):
        """Single training step."""
        real_sequences = real_sequences.to(self.device)
        labels = labels.to(self.device)
        batch_size, seq_len, _ = real_sequences.size()
        
        ae_loss, sup_loss, real_emb = self.autoencoder_step(real_sequences)

        for _ in range(n_critic):
            self.opt_D.zero_grad()
            
            z = torch.randn(batch_size, seq_len, self.noise_dim).to(self.device)
            fake_emb = self.generator(z, labels).detach()
            
            real_validity = self.discriminator(real_emb, labels)
            fake_validity = self.discriminator(fake_emb, labels)
            
            gradient_penalty = self.calculate_gradient_penalty(real_emb, fake_emb, labels, lambda_gp)
            
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
            
            d_loss.backward()
            self.opt_D.step()

        self.opt_G.zero_grad()
        
        z = torch.randn(batch_size, seq_len, self.noise_dim).to(self.device)
        fake_emb = self.generator(z, labels)
        
        fake_validity = self.discriminator(fake_emb, labels)
        g_loss = -torch.mean(fake_validity)

        self.supervisor.eval()
        for param in self.supervisor.parameters():
            param.requires_grad_(False)
        g_sup_loss = self.compute_supervisor_loss(fake_emb)
        for param in self.supervisor.parameters():
            param.requires_grad_(True)
        self.supervisor.train()

        total_g_loss = g_loss + (self.lambda_sup * g_sup_loss)
        total_g_loss.backward()
        self.opt_G.step()
        
        return ae_loss.item(), sup_loss.item(), d_loss.item(), g_loss.item(), g_sup_loss.item()

    def fit(
        self,
        dataloader,
        epochs: int = 1000,
        save_interval: int = 5,
        patience: int = 15,
        pretrain_epochs: int = 10,
        min_epochs: int = 50,
        min_delta: float = 1e-4
    ):
        """Main training loop."""
        self.load_checkpoint()
        
        metrics_file = os.path.join(self.checkpoint_dir, 'training_metrics.csv')
        if not os.path.exists(metrics_file):
            with open(metrics_file, 'w') as f:
                f.write("epoch,ae_loss,sup_loss,d_loss,g_loss,g_sup_loss\n")
                
        best_composite_score = float('inf')
        patience_counter = 0
        
        if pretrain_epochs > 0 and not self.start_epoch and dataloader is not None:
            logger.info(f"Pretraining autoencoder for {pretrain_epochs} epochs...")
            for epoch in range(pretrain_epochs):
                epoch_loss = 0.0
                for real_seqs, _ in tqdm(dataloader, desc=f"Pretrain {epoch + 1}/{pretrain_epochs}", leave=False):
                    real_seqs = real_seqs.to(self.device)
                    ae_loss, _, _ = self.autoencoder_step(real_seqs)
                    epoch_loss += ae_loss.item()
                logger.info(f"Pretrain Epoch {epoch + 1}: Loss={epoch_loss / len(dataloader):.4f}")

        logger.info("Starting C-TimeGAN Training...")
        try:
            for epoch in range(self.start_epoch, epochs):
                if dataloader is None:
                    time.sleep(0.5)
                    logger.info(f"Epoch {epoch + 1}/{epochs} (Mock mode)")
                else:
                    epoch_ae_loss = 0.0
                    epoch_sup_loss = 0.0
                    epoch_d_loss = 0.0
                    epoch_g_loss = 0.0
                    epoch_g_sup_loss = 0.0
                    
                    batch_iterator = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False)
                    
                    for batch_idx, (real_seqs, labels) in enumerate(batch_iterator):
                        ae_loss, sup_loss, d_loss, g_loss, g_sup_loss = self.train_step(real_seqs, labels)
                        
                        epoch_ae_loss += ae_loss
                        epoch_sup_loss += sup_loss
                        epoch_d_loss += d_loss
                        epoch_g_loss += g_loss
                        epoch_g_sup_loss += g_sup_loss
                        
                        batch_iterator.set_postfix({
                            "AE": f"{ae_loss:.4f}",
                            "Sup": f"{sup_loss:.4f}",
                            "D": f"{d_loss:.4f}",
                            "G": f"{g_loss:.4f}"
                        })
                    
                    avg_ae_loss = epoch_ae_loss / len(dataloader)
                    avg_sup_loss = epoch_sup_loss / len(dataloader)
                    avg_d_loss = epoch_d_loss / len(dataloader)
                    avg_g_loss = epoch_g_loss / len(dataloader)
                    avg_g_sup_loss = epoch_g_sup_loss / len(dataloader)
                    
                    logger.info(
                        f"[Epoch {epoch + 1}/{epochs}] AE: {avg_ae_loss:.4f} | "
                        f"Sup: {avg_sup_loss:.4f} | D: {avg_d_loss:.4f} | "
                        f"G: {avg_g_loss:.4f} | G_Sup: {avg_g_sup_loss:.4f}"
                    )
                    
                    with open(metrics_file, 'a') as f:
                        f.write(
                            f"{epoch + 1},{avg_ae_loss:.6f},{avg_sup_loss:.6f},"
                            f"{avg_d_loss:.6f},{avg_g_loss:.6f},{avg_g_sup_loss:.6f}\n"
                        )
                    
                    composite_score = avg_ae_loss + avg_sup_loss + avg_g_sup_loss
                    if composite_score < (best_composite_score - min_delta):
                        best_composite_score = composite_score
                        patience_counter = 0
                    else:
                        if (epoch + 1) >= min_epochs:
                            patience_counter += 1
                            if patience_counter >= patience:
                                logger.warning(f"Early stopping at Epoch {epoch + 1}")
                                self.save_checkpoint(epoch=epoch, is_final=True)
                                break
                
                if epoch > 0 and epoch % save_interval == 0:
                    self.save_checkpoint(epoch=epoch)
            
            if patience_counter < patience:
                self.save_checkpoint(epoch=epochs, is_final=True)
                logger.info("Training complete.")
        
        except KeyboardInterrupt:
            logger.warning(f"Training interrupted at Epoch {epoch}")
            self.save_checkpoint(epoch=epoch)
            logger.info("State saved. Resume training by re-running this cell.")

## Step 4: Data Configuration and Loading

In [ ]:
# ===== ENVIRONMENT-AWARE PATHS =====
# Automatically uses correct paths for Colab or RunPod

DATA_PATH = os.path.join(PROJECT_PATH, "data", "raw", "Combined.csv")
ARTIFACT_DIR = os.path.join(PROJECT_PATH, "models", "artifacts")
CHECKPOINT_DIR = os.path.join(PROJECT_PATH, "checkpoints")

logger.info(f"Project Path: {PROJECT_PATH}")
logger.info(f"Data Path: {DATA_PATH}")
logger.info(f"Artifact Dir: {ARTIFACT_DIR}")
logger.info(f"Checkpoint Dir: {CHECKPOINT_DIR}")

# Verify data exists
if os.path.exists(DATA_PATH):
    logger.info(f"✓ Data file found: {os.path.getsize(DATA_PATH) / (1024**2):.2f} MB")
else:
    logger.warning(f"⚠ Data file not found at {DATA_PATH}")

In [ ]:
# ===== DATA PIPELINE: Import preprocessing utilities =====
from data_parser import NIDDParser
from preprocessor import DataPreprocessor
from sequence_generator import SequenceGenerator as DataSequenceGenerator

logger.info("Loading and preprocessing data...")

# 1. Parse Data
parser = NIDDParser(DATA_PATH)
df = parser.load_data()
logger.info(f"Loaded data shape: {df.shape}")

# 2. Sample data for memory safety (adjust SAMPLE_SIZE for your GPU)
SAMPLE_SIZE = 100000  # Change to len(df) to use full dataset
df_sample = df.head(SAMPLE_SIZE).copy()
logger.info(f"Using sample of {len(df_sample)} rows")

# 3. Preprocess (Scale & Encode)
preprocessor = DataPreprocessor()
df_processed = preprocessor.fit_transform(df_sample)

# Save preprocessing artifacts for later reverse transformation
os.makedirs(ARTIFACT_DIR, exist_ok=True)
preprocessor.save_pipeline(ARTIFACT_DIR)
logger.info(f"Saved preprocessing artifacts to {ARTIFACT_DIR}")

# 4. Generate Sequences
seq_gen = DataSequenceGenerator(sequence_length=20, stride=5)
X, y = seq_gen.create_windows(df_processed, label_col='Attack Type')
logger.info(f"Generated sequences: X shape {X.shape}, y shape {y.shape}")

# 5. Get feature metadata
feature_cols = preprocessor.feature_columns
categorical_cols = preprocessor.categorical_cols
continuous_cols = preprocessor.continuous_cols
categorical_indices = [feature_cols.index(col) for col in categorical_cols]
continuous_indices = [feature_cols.index(col) for col in continuous_cols]
categorical_sizes = [len(preprocessor.categorical_encoders[col].classes_) for col in categorical_cols]

logger.info(f"Feature dimensions: {len(feature_cols)} features")
logger.info(f"Categorical features: {len(categorical_indices)}")
logger.info(f"Continuous features: {len(continuous_indices)}")

# 6. Convert to PyTorch tensors and create DataLoader
X_tensor = torch.FloatTensor(X)
y_tensor = torch.LongTensor(y)

dataset = TensorDataset(X_tensor, y_tensor)
BATCH_SIZE = 64  # Adjust based on GPU memory
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

logger.info(f"✓ Dataloader ready: {len(dataloader)} batches of size {BATCH_SIZE}")

# Print summary
logger.info("\n" + "="*60)
logger.info("DATA PIPELINE SUMMARY")
logger.info("="*60)
logger.info(f"Raw data shape: {df.shape}")
logger.info(f"Sample used: {len(df_sample)} / {len(df)} rows")
logger.info(f"Processed features: {len(feature_cols)}")
logger.info(f"  - Continuous: {continuous_cols}")
logger.info(f"  - Categorical: {categorical_cols}")
logger.info(f"Sequence parameters: length=20, stride=5")
logger.info(f"Total sequences: {X.shape[0]}")
logger.info(f"Dataloader batches: {len(dataloader)}")
logger.info("="*60)

## Step 5: Training Configuration

In [ ]:
# ===== TRAINING HYPERPARAMETERS =====
num_attack_types = len(preprocessor.categorical_encoders['Attack Type'].classes_)

TRAINING_CONFIG = {
    # Model Architecture (auto-configured from data)
    'noise_dim': 16,
    'hidden_dim': 24,
    'input_dim': len(feature_cols),
    'num_classes': num_attack_types,
    'lr': 1e-4,
    
    # Loss Weights
    'lambda_cat': 1.0,      # Categorical reconstruction weight
    'lambda_sup': 1.0,      # Supervisor loss weight
    'lambda_gp': 10.0,      # Gradient penalty weight
    
    # Training Loop
    'epochs': 500,          # Total epochs (will use early stopping)
    'batch_size': 64,       # Adjust based on GPU memory
    'save_interval': 5,     # Save checkpoint every N epochs
    'n_critic': 3,          # Discriminator updates per generator update
    
    # Early Stopping
    'patience': 15,         # Stop if no improvement for N epochs
    'min_epochs': 50,       # Minimum epochs before early stopping
    'min_delta': 1e-4,      # Minimum improvement threshold
    
    # Pretraining
    'pretrain_epochs': 10,  # Pretrain autoencoder first
}

logger.info("Training Configuration (auto-configured from data):")
for key, value in TRAINING_CONFIG.items():
    logger.info(f"  {key}: {value}")

## Step 6: Initialize and Train

In [ ]:
# Initialize trainer with actual data dimensions
logger.info("Initializing trainer...")
trainer = RobustTrainer(
    noise_dim=TRAINING_CONFIG['noise_dim'],
    hidden_dim=TRAINING_CONFIG['hidden_dim'],
    input_dim=TRAINING_CONFIG['input_dim'],
    num_classes=TRAINING_CONFIG['num_classes'],
    lr=TRAINING_CONFIG['lr'],
    continuous_indices=continuous_indices,
    categorical_indices=categorical_indices,
    categorical_sizes=categorical_sizes,
    lambda_cat=TRAINING_CONFIG['lambda_cat'],
    lambda_sup=TRAINING_CONFIG['lambda_sup'],
    checkpoint_dir=CHECKPOINT_DIR
)

logger.info("✓ Trainer initialized successfully!")
logger.info(f"Total model parameters:")
total_params = sum(p.numel() for p in trainer.autoencoder.parameters())
total_params += sum(p.numel() for p in trainer.generator.parameters())
total_params += sum(p.numel() for p in trainer.discriminator.parameters())
total_params += sum(p.numel() for p in trainer.supervisor.parameters())
logger.info(f"  Total: {total_params:,} parameters")
logger.info(f"  Input shape per batch: {TRAINING_CONFIG['batch_size']} x 20 x {TRAINING_CONFIG['input_dim']}")
logger.info(f"  Number of attack types: {TRAINING_CONFIG['num_classes']}")

In [ ]:
# Start training with live GPU and loss monitoring
import threading
import subprocess
import matplotlib.pyplot as plt
from IPython.display import clear_output

logger.info("\n" + "=" * 50)
logger.info("Starting C-TimeGAN Training with Live Monitoring")
logger.info("=" * 50 + "\n")

metrics_file = os.path.join(CHECKPOINT_DIR, 'training_metrics.csv')


def get_gpu_stats():
    """Return GPU utilization and memory stats (best-effort)."""
    if not torch.cuda.is_available():
        return {
            'util': 0.0,
            'mem_used': 0.0,
            'mem_total': 0.0,
            'alloc': 0.0,
            'reserved': 0.0,
            'name': 'CPU'
        }

    alloc_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)
    device_name = torch.cuda.get_device_name(0)

    util = 0.0
    mem_used = reserved_gb
    mem_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    try:
        cmd = [
            'nvidia-smi',
            '--query-gpu=utilization.gpu,memory.used,memory.total',
            '--format=csv,noheader,nounits'
        ]
        out = subprocess.check_output(cmd, text=True).strip().splitlines()[0]
        gpu_util, used_mb, total_mb = [float(x.strip()) for x in out.split(',')]
        util = gpu_util
        mem_used = used_mb / 1024.0
        mem_total = total_mb / 1024.0
    except Exception:
        # Fallback to torch memory info if nvidia-smi is unavailable
        pass

    return {
        'util': util,
        'mem_used': mem_used,
        'mem_total': mem_total,
        'alloc': alloc_gb,
        'reserved': reserved_gb,
        'name': device_name
    }


training_error = {'err': None}


def run_training():
    try:
        trainer.fit(
            dataloader=dataloader,
            epochs=TRAINING_CONFIG['epochs'],
            save_interval=TRAINING_CONFIG['save_interval'],
            patience=TRAINING_CONFIG['patience'],
            pretrain_epochs=TRAINING_CONFIG['pretrain_epochs'],
            min_epochs=TRAINING_CONFIG['min_epochs'],
            min_delta=TRAINING_CONFIG['min_delta']
        )
    except Exception as e:
        training_error['err'] = e


start_time = time.time()
train_thread = threading.Thread(target=run_training, daemon=True)
train_thread.start()

# Time series for GPU monitor plot
timestamps = []
gpu_utils = []
gpu_mem_used = []
gpu_mem_total = None

while train_thread.is_alive():
    stats = get_gpu_stats()
    elapsed_min = (time.time() - start_time) / 60.0

    timestamps.append(elapsed_min)
    gpu_utils.append(stats['util'])
    gpu_mem_used.append(stats['mem_used'])
    gpu_mem_total = stats['mem_total']

    df_metrics = None
    if os.path.exists(metrics_file):
        try:
            df_metrics = pd.read_csv(metrics_file)
        except Exception:
            df_metrics = None

    clear_output(wait=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Plot 1: Training losses by epoch
    ax1 = axes[0]
    if df_metrics is not None and len(df_metrics) > 0:
        ax1.plot(df_metrics['epoch'], df_metrics['ae_loss'], label='AE')
        ax1.plot(df_metrics['epoch'], df_metrics['sup_loss'], label='Sup')
        ax1.plot(df_metrics['epoch'], df_metrics['d_loss'], label='D')
        ax1.plot(df_metrics['epoch'], df_metrics['g_loss'], label='G')
        ax1.plot(df_metrics['epoch'], df_metrics['g_sup_loss'], label='G_Sup')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training Loss (updates each epoch)')
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)
    else:
        ax1.text(0.5, 0.5, 'Waiting for first epoch metrics...', ha='center', va='center')
        ax1.set_title('Training Loss')
        ax1.set_xticks([])
        ax1.set_yticks([])

    # Plot 2: GPU utilization + memory over time
    ax2 = axes[1]
    ax2.plot(timestamps, gpu_utils, color='tab:orange', label='GPU Util %')
    ax2.set_xlabel('Elapsed Time (min)')
    ax2.set_ylabel('Utilization (%)', color='tab:orange')
    ax2.tick_params(axis='y', labelcolor='tab:orange')
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)

    ax2b = ax2.twinx()
    ax2b.plot(timestamps, gpu_mem_used, color='tab:blue', label='GPU Mem (GB)')
    ax2b.set_ylabel('Memory (GB)', color='tab:blue')
    ax2b.tick_params(axis='y', labelcolor='tab:blue')
    if gpu_mem_total and gpu_mem_total > 0:
        ax2b.set_ylim(0, gpu_mem_total * 1.05)

    ax2.set_title('GPU Monitoring (live)')

    plt.tight_layout()
    plt.show()

    # Console status summary
    logger.info(
        f"Device: {stats['name']} | Util: {stats['util']:.1f}% | "
        f"Mem: {stats['mem_used']:.2f}/{stats['mem_total']:.2f} GB | "
        f"Torch alloc/reserved: {stats['alloc']:.2f}/{stats['reserved']:.2f} GB"
    )

    time.sleep(2)

# Ensure thread completion
train_thread.join()

# Final redraw after training completes
stats = get_gpu_stats()
df_metrics = pd.read_csv(metrics_file) if os.path.exists(metrics_file) else None

clear_output(wait=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
if df_metrics is not None and len(df_metrics) > 0:
    ax1.plot(df_metrics['epoch'], df_metrics['ae_loss'], label='AE')
    ax1.plot(df_metrics['epoch'], df_metrics['sup_loss'], label='Sup')
    ax1.plot(df_metrics['epoch'], df_metrics['d_loss'], label='D')
    ax1.plot(df_metrics['epoch'], df_metrics['g_loss'], label='G')
    ax1.plot(df_metrics['epoch'], df_metrics['g_sup_loss'], label='G_Sup')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Final Training Loss Curves')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No metrics found.', ha='center', va='center')
    ax1.set_xticks([])
    ax1.set_yticks([])

ax2 = axes[1]
ax2.plot(timestamps, gpu_utils, color='tab:orange', label='GPU Util %')
ax2.set_xlabel('Elapsed Time (min)')
ax2.set_ylabel('Utilization (%)', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3)

ax2b = ax2.twinx()
ax2b.plot(timestamps, gpu_mem_used, color='tab:blue', label='GPU Mem (GB)')
ax2b.set_ylabel('Memory (GB)', color='tab:blue')
ax2b.tick_params(axis='y', labelcolor='tab:blue')
if gpu_mem_total and gpu_mem_total > 0:
    ax2b.set_ylim(0, gpu_mem_total * 1.05)

ax2.set_title('Final GPU Usage Timeline')

plt.tight_layout()
plt.show()

elapsed_time = time.time() - start_time
logger.info(f"\nTraining completed in {elapsed_time/3600:.2f} hours")
logger.info(
    f"Final device stats -> {stats['name']}: util={stats['util']:.1f}%, "
    f"mem={stats['mem_used']:.2f}/{stats['mem_total']:.2f} GB"
)

if training_error['err'] is not None:
    raise training_error['err']

## Step 7: Post-Training Utilities

In [ ]:
# View training metrics
metrics_file = os.path.join(CHECKPOINT_DIR, 'training_metrics.csv')
if os.path.exists(metrics_file):
    metrics_df = pd.read_csv(metrics_file)
    logger.info("\nTraining Metrics Summary:")
    logger.info(metrics_df.tail(10).to_string())
    logger.info(f"\nTotal epochs trained: {len(metrics_df)}")
else:
    logger.warning("No metrics file found.")

In [ ]:
# List all saved checkpoints
import glob

checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, '*.pt'))
if checkpoints:
    logger.info("\nSaved Checkpoints:")
    for ckpt in sorted(checkpoints):
        size_mb = os.path.getsize(ckpt) / (1024 * 1024)
        logger.info(f"  {os.path.basename(ckpt)} ({size_mb:.2f} MB)")
else:
    logger.warning("No checkpoints found.")

## Optional: Generate Synthetic Data

In [ ]:
def generate_synthetic_sequences(trainer, num_sequences=1000, seq_len=20):
    """
    Generate synthetic sequences using the trained generator.
    Returns raw latent embeddings from the model.
    """
    trainer.generator.eval()
    
    synthetic_data = []
    synthetic_labels = []
    
    with torch.no_grad():
        num_batches = (num_sequences // 64) + 1
        for i in range(num_batches):
            batch_size = min(64, num_sequences - len(synthetic_data))
            z = torch.randn(batch_size, seq_len, trainer.noise_dim).to(trainer.device)
            labels = torch.randint(0, TRAINING_CONFIG['num_classes'], (batch_size,)).to(trainer.device)
            
            fake_emb = trainer.generator(z, labels)
            synthetic_data.append(fake_emb.cpu().numpy())
            synthetic_labels.append(labels.cpu().numpy())
    
    synthetic_array = np.vstack(synthetic_data)[:num_sequences]
    synthetic_labels_array = np.hstack(synthetic_labels)[:num_sequences]
    
    logger.info(f"✓ Generated {synthetic_array.shape[0]} synthetic sequences")
    logger.info(f"  Shape: {synthetic_array.shape}")
    logger.info(f"  Label distribution: {np.unique(synthetic_labels_array, return_counts=True)}")
    
    return synthetic_array, synthetic_labels_array

# Example usage:
# logger.info("\nGenerating synthetic data...")
# synthetic_seqs, synthetic_labels = generate_synthetic_sequences(trainer, num_sequences=1000)
# 
# # Save synthetic sequences
# synthetic_path = os.path.join(CHECKPOINT_DIR, "synthetic_sequences.npy")
# np.save(synthetic_path, synthetic_seqs)
# logger.info(f"Saved synthetic sequences to {synthetic_path}")